# Untersuchung nicht referenzgedeckter URL- und Secret-Vorhersagen

Dieses Notebook untersucht die im Entity-Benchmark als False Positives gezählten `PRIVATE_URL`- und `SECRET`-Vorhersagen des ONNX-INT8 Privacy Filters sowie die nicht referenzgedeckten `PRIVATE_URL`-Vorhersagen von Presidio. Es trennt drei Fälle: Vorhersagen mit Überlappung zu einer anders gelabelten OpenPII-Annotation, URL-ähnliche aber nicht annotierte Textstellen und übrige, manuell zu prüfende Textstellen.

Die Auswertung folgt bewusst derselben Definition wie die Benchmark-Skripte: Eine Vorhersage ist nur dann referenzgedeckt, wenn sie einen Referenzspan **derselben Vergleichsklasse** überlappt.

In [1]:
from __future__ import annotations

import ast
import re
from collections import Counter
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoConfig

import benchmark_privacy_filter_onnx_int8_Entity_level as pf_entity
import benchmark_presidio_de_core_news_lg_Entity_level as presidio_entity
from benchmark_privacy_filter_onnx_int8 import (
    build_onnx_int8_session,
    load_tokenizer_robust,
    load_validation_dataset,
)

DATASET = 'ai4privacy/pii-masking-openpii-1m'
SPLIT = 'validation'
LANGUAGE = 'de'
MAX_SAMPLES = 4000  # begrenzte, reproduzierbare Stichprobe aus dem deutschen Validierungssplit
CONTEXT_CHARS = 90
OUTPUT_DIR = Path('unmatched_prediction_investigation')
OUTPUT_DIR.mkdir(exist_ok=True)

URL_PATTERN = re.compile(r'(?i)(?:https?://|www\.)[^\s<>]+|\b[a-z0-9][a-z0-9.-]*\.[a-z]{2,}(?:/[^\s<>]*)?')
TARGETS = {'PRIVATE_URL', 'SECRET'}


c:\Users\jan.wobker\Desktop\pii_detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def raw_reference_spans(privacy_mask):
    """Liest alle Originalannotation, auch die im Benchmark nicht unterstützten."""
    if isinstance(privacy_mask, str):
        privacy_mask = ast.literal_eval(privacy_mask)
    spans = []
    for item in privacy_mask or []:
        if not isinstance(item, dict):
            continue
        try:
            start, end = int(item['start']), int(item['end'])
        except (KeyError, TypeError, ValueError):
            continue
        _, original_label = pf_entity.strip_bio(item.get('label'))
        mapped_label = pf_entity.map_label(original_label, model_label=False)
        spans.append((start, end, original_label, mapped_label))
    return spans

def overlaps(start, end, other_start, other_end):
    return start < other_end and other_start < end

def classify_unmatched(text, entity, reference_spans):
    overlapping = [span for span in reference_spans if overlaps(entity.start, entity.end, span[0], span[1])]
    original_labels = sorted({span[2] for span in overlapping})
    if overlapping:
        return 'overlaps_other_reference_label', ', '.join(original_labels)
    predicted_text = text[entity.start:entity.end]
    if entity.entity_type == 'PRIVATE_URL' and URL_PATTERN.search(predicted_text):
        return 'url_like_without_reference', ''
    return 'no_reference_overlap_manual_review', ''

def context(text, start, end, width=CONTEXT_CHARS):
    return text[max(0, start - width):min(len(text), end + width)]

def record(system, row_id, text, entity, reference_spans):
    category, overlap_labels = classify_unmatched(text, entity, reference_spans)
    return {
        'system': system,
        'row_id': row_id,
        'entity_type': entity.entity_type,
        'start': entity.start,
        'end': entity.end,
        'predicted_text': text[entity.start:entity.end],
        'category': category,
        'overlapping_original_labels': overlap_labels,
        'context': context(text, entity.start, entity.end),
    }


In [3]:
dataset = load_validation_dataset(DATASET, SPLIT)
dataset = dataset.filter(lambda row: str(row.get('language', '')).lower() == LANGUAGE)
if MAX_SAMPLES:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))
print(f'Untersuchte Dokumente: {len(dataset):,}')

# Privacy Filter: dieselbe ONNX-INT8-Inferenz wie im Benchmark.
tokenizer = load_tokenizer_robust('openai/privacy-filter')
session = build_onnx_int8_session('openai/privacy-filter')
config = AutoConfig.from_pretrained('openai/privacy-filter', trust_remote_code=True)
id2label = {int(key): value for key, value in config.id2label.items()}

privacy_filter_records = []
batch_size = 8
for batch_start in tqdm(range(0, len(dataset), batch_size), desc='Privacy Filter'):
    rows = [dataset[i] for i in range(batch_start, min(batch_start + batch_size, len(dataset)))]
    texts = [row.get('source_text') or '' for row in rows]
    predictions = pf_entity.predict_entities_onnx(texts, tokenizer, session, id2label)
    for offset, (row, (entities, _)) in enumerate(zip(rows, predictions)):
        text = texts[offset]
        references = raw_reference_spans(row.get('privacy_mask'))
        for entity in set(entities):
            if entity.entity_type not in TARGETS:
                continue
            # Gleiche Regel wie update_metrics: nur typgleiche Überlappung deckt ab.
            is_covered = any(
                mapped == entity.entity_type and overlaps(entity.start, entity.end, start, end)
                for start, end, _, mapped in references
            )
            if not is_covered:
                privacy_filter_records.append(record('privacy_filter_onnx_int8', batch_start + offset, text, entity, references))

pd.DataFrame(privacy_filter_records).to_csv(OUTPUT_DIR / 'privacy_filter_unmatched.csv', index=False)
pd.DataFrame(privacy_filter_records).groupby(['entity_type', 'category']).size().rename('count').reset_index()


Untersuchte Dokumente: 4,000


Privacy Filter: 100%|██████████| 500/500 [26:06<00:00,  3.13s/it]


,entity_type,category,count
0,PRIVATE_URL,no_reference_overlap_manual_review,1
1,PRIVATE_URL,overlaps_other_reference_label,5
2,PRIVATE_URL,url_like_without_reference,1
3,SECRET,no_reference_overlap_manual_review,1
4,SECRET,overlaps_other_reference_label,29


In [4]:
# Presidio: identische Modell- und Schwellenkonfiguration wie im Benchmark.
analyzer = presidio_entity.build_analyzer()
presidio_records = []
for row_id, row in enumerate(tqdm(dataset, desc='Presidio')):
    text = row.get('source_text') or ''
    entities, _ = presidio_entity.predict_entities(analyzer, text, threshold=0.35)
    references = raw_reference_spans(row.get('privacy_mask'))
    for entity in set(entities):
        if entity.entity_type != 'PRIVATE_URL':
            continue
        is_covered = any(
            mapped == entity.entity_type and overlaps(entity.start, entity.end, start, end)
            for start, end, _, mapped in references
        )
        if not is_covered:
            presidio_records.append(record('presidio_de_core_news_lg', row_id, text, entity, references))

pd.DataFrame(presidio_records).to_csv(OUTPUT_DIR / 'presidio_unmatched_urls.csv', index=False)
pd.DataFrame(presidio_records).groupby(['entity_type', 'category']).size().rename('count').reset_index()


Presidio: 100%|██████████| 4000/4000 [01:56<00:00, 34.30it/s]


,entity_type,category,count
0,PRIVATE_URL,overlaps_other_reference_label,2199
1,PRIVATE_URL,url_like_without_reference,19


In [5]:
all_records = pd.DataFrame(privacy_filter_records + presidio_records)
summary = (
    all_records.groupby(['system', 'entity_type', 'category'])
    .size().rename('count').reset_index()
    .sort_values(['system', 'entity_type', 'count'], ascending=[True, True, False])
)
summary.to_csv(OUTPUT_DIR / 'summary.csv', index=False)
summary


,system,entity_type,category,count
0,presidio_de_core_news_lg,PRIVATE_URL,overlaps_other_reference_label,2199
1,presidio_de_core_news_lg,PRIVATE_URL,url_like_without_reference,19
3,privacy_filter_onnx_int8,PRIVATE_URL,overlaps_other_reference_label,5
2,privacy_filter_onnx_int8,PRIVATE_URL,no_reference_overlap_manual_review,1
4,privacy_filter_onnx_int8,PRIVATE_URL,url_like_without_reference,1
6,privacy_filter_onnx_int8,SECRET,overlaps_other_reference_label,29
5,privacy_filter_onnx_int8,SECRET,no_reference_overlap_manual_review,1


## Fachliche Stichprobenprüfung

Die Heuristiken liefern nur eine Vorsortierung. Für eine belastbare Aussage sollten zufällig gezogene Beispiele pro Kategorie geprüft werden. Ein URL-ähnlicher, nicht annotierter Text kann eine Annotation-Lücke sein; ebenso kann er ein bewusst nicht schutzwürdiger Link sein.

In [6]:
SAMPLES_PER_GROUP = 20
review_sample = (
    all_records.groupby(['system', 'entity_type', 'category'], group_keys=False)
    .apply(lambda frame: frame.sample(min(SAMPLES_PER_GROUP, len(frame)), random_state=42))
    .sort_values(['system', 'entity_type', 'category'])
)
review_sample.to_csv(OUTPUT_DIR / 'manual_review_sample.csv', index=False)
review_sample[['system', 'entity_type', 'category', 'predicted_text', 'overlapping_original_labels', 'context']]


KeyError: 'system'